# OpenCode with Ollama in Colab



## Setup




### Install packages


In [ ]:
%%capture output_system_install
%%bash

# bash setup
fgrep -q '.bash_aliases' ~/.bashrc || {
  echo -e '\n\n\n[ -f ~/.bash_aliases ] && source ~/.bash_aliases\n' >> ~/.bashrc
}

{ cat << 'eof'
alias cls='clear'
alias dir='ls -la'
alias goTo='cd '
alias goUp='cd ..'
alias whereAmI='pwd'

alias me.group.id='id -g'
alias me.group.name='id -g -n'
alias me.groups='id -G'
alias me.groups.ids='id -G'
alias me.groups.names='id -G -n '
alias me.id='id -u'
alias me.name='id -u -n'

eof
} > ~/.bash_aliases

{ cat << 'eof'

export PATH='/root/.local/bin':$PATH
eof
} >> ~/.bashrc


# system package installs
tmux new -s update -d " \
  apt-get update ;\
  apt-get install -y zstd ;\
  apt-get install -y tree jq ncal less texlive-xetex pandoc ; \
  echo == Done ; \
  sleep 30
"


### Jupyter

In [ ]:
%%capture output_install_run_jupyter
%%bash

# jupyter install
tmux new -s jupyter-server -d " \
  pip install ipyaml jupyterlab ; \
  jupyter labextension disable @jupyterlab/apputils-extension:announcements ; \
  jupyter lab \
    --ip=127.0.0.1 \
    --port=8888 \
    --no-browser \
    --allow-root \
    --NotebookApp.token='' ; \
  echo == Done ; \
  sleep 30
"


### Claude Code

In [ ]:
%%capture output_install_claude_code
%%bash

# claude code install
until which zstd ; do date ; sleep 1 ;done
curl -fsSL https://claude.ai/install.sh | bash
echo 'export PATH='/root/.local/bin':$PATH' >> ~/.bashrc


### Open Code


In [ ]:
%%capture output_install_open_code
%%bash

# opencode install
until which zstd ; do date ; sleep 1 ;done
curl -fsSL https://opencode.ai/install | bash


### Ollama


In [ ]:
%%capture output_install_run_ollama
%%bash

# ollama service install and launch
tmux new -s ollama -d "\
  mkdir /tmp/ollama-logs/ ; \
  exec > /tmp/ollama-logs/ollama.log 2>&1 ; \
  while ! which zstd ; do sleep 1 ;done ; \
  curl -fsSL https://ollama.com/install.sh | sh ; \
  OLLAMA_KEEP_ALIVE=20m OLLAMA_FLASH_ATTENTION=1 ollama serve ; \
  echo == Done ; \
  sleep 10
"


### Ollama models


Create a model file to modify the existing model


In [ ]:
%%writefile Modelfile.qwen
FROM qwen2.5-coder:7b-instruct-q8_0
PARAMETER num_ctx 32768


In [ ]:
%%writefile Modelfile.gemma4
FROM gemma4:e4b
PARAMETER num_ctx 32768


In [ ]:
%%bash
mkdir -p ~/.config/opencode
cat <<'eof' > ~/.config/opencode/opencode.json
{
  "$schema": "https://opencode.ai/config.json",
  "provider": {
    "ollama": {
      "npm": "ollama-ai-provider-v2",
      "name": "Ollama",
      "options": {
        "baseURL": "http://localhost:11434/api"
      },
      "models": {
        "llama3.1:8b": {
          "name": "llama3.1:8b"
        },
        "qwen2.5-coder:7b-32k": {
          "name": "qwen2.5-coder:7b-32k"
        },
        "gemma4:e4b": {
          "name": "gemma4:e4b",
            "options": {
              "num_ctx": 32768,
              "temperature": 0.3
            }
        },
        "gemma4:e4b-32k": {
          "name": "gemma4:e4b-32k",
            "options": {
              "temperature": 0.3
            }
        },
       "gemma4:12b": {
          "name": "gemma4:12b"
        }
      }
    }
  },
  "permission": {
    "edit": "allow",
    "bash": "ask"
  }
}
eof


In [ ]:
%%capture output_install_ollama_models
%%bash

# ollama models pull and load
tmux new -s ollama_models -d "\
  mkdir /tmp/ollama-logs/ ; \
  exec > /tmp/ollama-logs/ollama.models.log 2>&1 ; \
  while ! curl -s -I 127.0.0.1:11434 ; do date ; sleep 1 ; done ;\
  ollama pull qwen2.5-coder:7b-instruct-q8_0 ; \
  ollama create qwen2.5-coder:7b-32k -f Modelfile.qwen ;\
  ollama create gemma4:e4b-32k -f Modelfile.gemma4 ;\
  ollama run qwen2.5-coder:7b-32k --keepalive 20m "" ;\
  ollama ps ; \
  echo llama3.1:8b gemma4:e4b gemma4:12b  | nice -n 19 ionice -c 3 xargs -n 1 -P 5 ollama pull ; \
  echo ; \
  echo == Done ; \
  sleep 10
"


## Modules, etc.


In [ ]:
%alias tree tree

from datetime import datetime, timezone
from time import sleep
from google.colab import output
import requests

print("Waiting for Jupyter to start")
for i in range(300):
  try:
    requests.head( "http://127.0.0.1:8888" )
    print()
    break
  except:
    print("=", end="")
  sleep(1)
print(f"{i} seconds")

print("Jupyter has started")
output.serve_kernel_port_as_window(8888)


Wait for Ollama to download a model


In [ ]:
%%bash
echo Waiting for a model
until ollama list | grep -q qwen2.5-coder:7b-32k ; do sleep 1 ; done


## Using Open Code and Ollama


When ready, click on the link to Jupyter Lab, open a terminal, and type this to start opencode:

```
opencode run -m ollama/qwen2.5-coder:7b-32k "what is the capital of France?"
```
Or you can run it here.


In [ ]:
!/root/.opencode/bin/opencode run -m ollama/qwen2.5-coder:7b-32k "what is the capital of France?"


or ... type this to interact with ollama chat:

```
ollama run qwen2.5-coder:7b-32k "what is the capital of France?"
```

Or you can run it here.






In [ ]:
!ollama run qwen2.5-coder:7b-32k "what is the capital of France?"

## Timer 1


In [ ]:
# show a timer for 30 minutes
print("Timer 1")
for i in range(60*30):
  utc_now = datetime.now(timezone.utc)
  print(f"\r{utc_now.timetz().isoformat(timespec='seconds')} {'==' * (utc_now.second % 10)}", end='')
  sleep(1)


## Python tool calling


In [ ]:
%%bash
pip install ollama --break-system-packages


In [ ]:
import ollama
import os


### Using qwen2.5

In [ ]:
def write_file(filepath: str, content: str) -> str:
    """Write content to a file at the given path.

    Args:
        filepath: The path where the file should be created.
        content: The text content to write into the file.
    """
    with open(filepath, 'w') as f:
        f.write(content)
    return f"File written successfully to {filepath}"

available_functions = {'write_file': write_file}


In [ ]:
response = ollama.chat(
    model='qwen2.5-coder:14b-instruct-q4_K_M',
    messages=[{
        'role': 'user',
        'content': 'Create a file called hello.py with a simple hello world script in python'
    }],
    tools=[write_file],
)
response


In [ ]:
dict(response["message"])


In [ ]:
import json
for tool in [{"function" : json.loads(response.message.content) }] or []:
    fn = available_functions.get(tool["function"]["name"])
    if fn:
        result = fn(**tool["function"]["arguments"])
        print(result)
    else:
        print('Unknown function:', tool["function"]["name"])


In [ ]:
!ls -la


In [ ]:
!cat -n hello.py

#### Conclusion - don't use

Something is wonky with the tool caling formatting.


### Using llama3.1


In [ ]:
!rm hello.py
!ls -l


Quick test


In [ ]:
! ollama list


In [ ]:
! ollama show llama3.1:8b


In [ ]:
import ollama

response = ollama.chat(
    model='llama3.1:8b',  # or whichever you test
    messages=[{'role': 'user', 'content': "What's the weather in Austin?"}],
    tools=[{"type": "function", "function": {
        "name": "get_weather",
        "description": "Get current weather for a city",
        "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}
    }}],
)
print(response.message.tool_calls)  # should NOT be None if the model's template handles this properly


Create 'hello.py'


In [ ]:
import ollama

def write_file(filepath: str, content: str) -> str:
    """Write content to a file at the given path.

    Args:
        filepath: The path where the file should be created.
        content: The text content to write into the file.
    """
    with open(filepath, 'w') as f:
        f.write(content)
    return f"File written successfully to {filepath}"

available_functions = {'write_file': write_file}

response = ollama.chat(
    model='llama3.1:8b',
    messages=[{
        'role': 'user',
        'content': 'Create a file called hello.py with a simple hello world script in python'
    }],
    tools=[write_file],
)

print(response.message.tool_calls)

for tool in response.message.tool_calls or []:
    fn = available_functions.get(tool.function.name)
    if fn:
        result = fn(**tool.function.arguments)
        print(result)




In [ ]:
!cat -n hello.py


### Using gemma4:e4b


In [ ]:
!rm hello.py
!ls -l


Quick test


In [ ]:
! ollama list


In [ ]:
! ollama show gemma4:e4b


In [ ]:
import ollama

response = ollama.chat(
    model='gemma4:e4b',  # or whichever you test
    messages=[{'role': 'user', 'content': "What's the weather in Austin?"}],
    tools=[{"type": "function", "function": {
        "name": "get_weather",
        "description": "Get current weather for a city",
        "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}
    }}],
)
print(response.message.tool_calls)  # should NOT be None if the model's template handles this properly


Create 'hello.py'


In [ ]:
import ollama

def write_file(filepath: str, content: str) -> str:
    """Write content to a file at the given path.

    Args:
        filepath: The path where the file should be created.
        content: The text content to write into the file.
    """
    with open(filepath, 'w') as f:
        f.write(content)
    return f"File written successfully to {filepath}"

available_functions = {'write_file': write_file}

response = ollama.chat(
    model='gemma4:e4b',
    messages=[{
        'role': 'user',
        'content': 'Create a file called hello.py with a simple hello world script in python'
    }],
    tools=[write_file],
)

print(response.message.tool_calls)

for tool in response.message.tool_calls or []:
    fn = available_functions.get(tool.function.name)
    if fn:
        result = fn(**tool.function.arguments)
        print(result)




In [ ]:
!cat -n hello.py


### Using gemma4:12b


In [ ]:
!rm hello.py
!ls -la


Quick test


In [ ]:
! ollama list


In [ ]:
! ollama show gemma4:12b


In [ ]:
import ollama

response = ollama.chat(
    model='gemma4:12b',  # or whichever you test
    messages=[{'role': 'user', 'content': "What's the weather in Austin?"}],
    tools=[{"type": "function", "function": {
        "name": "get_weather",
        "description": "Get current weather for a city",
        "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}
    }}],
)
print(response.message.tool_calls)  # should NOT be None if the model's template handles this properly


Create 'hello.py'


In [ ]:
import ollama

def write_file(filepath: str, content: str) -> str:
    """Write content to a file at the given path.

    Args:
        filepath: The path where the file should be created.
        content: The text content to write into the file.
    """
    with open(filepath, 'w') as f:
        f.write(content)
    return f"File written successfully to {filepath}"

available_functions = {'write_file': write_file}

response = ollama.chat(
    model='gemma4:12b',
    messages=[{
        'role': 'user',
        'content': 'Create a file called hello.py with a simple hello world script in python'
    }],
    tools=[write_file],
)

print(response.message.tool_calls)

for tool in response.message.tool_calls or []:
    fn = available_functions.get(tool.function.name)
    if fn:
        result = fn(**tool.function.arguments)
        print(result)




In [ ]:
!cat -n hello.py


## Attach Googe Drive for data


In [ ]:
from google.colab import drive
drive.mount(
  '/content/drive',
  readonly=True,
)